# Notebook 09: Feature Engineering

The raw predictors in model_dataset_v3 are ready for cleaning and transformation 
before modeling. This notebook does four things.

First, it resolves the remaining data quality issue: 292 None values in the wealth 
index column belonging to Dzaleka records. These are imputed with the district-level 
mode so they do not create null rows in the national model while remaining 
distinguishable through the flag_dzaleka column.

Second, it engineers new features from existing variables. Child age in months becomes 
age bands aligned to the biological windows of the first 1000 days. Maternal BMI is 
derived from weight and height. Birth interval gets a short interval flag below 24 
months which is the WHO threshold for increased child mortality and stunting risk. 
Religion categories are collapsed from nine to five to reduce sparsity.

Third, it encodes all categorical variables into model-ready format using ordinal 
encoding for ordered categories and label encoding for nominal ones. This notebook 
does not one-hot encode because tree-based models handle label encoding natively and 
we want to keep the dataset compact for the SHAP analysis.

Fourth, it creates the final national analysis dataset by filtering out Dzaleka 
records, which the 2024 MDHS sampling design excludes from national estimates.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT   = Path("/Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(DATA_PROCESSED / "model_dataset_v3.parquet")

print("Shape:", df.shape)
print("Dzaleka records:", df["flag_dzaleka"].sum())
print("National records:", (df["flag_dzaleka"] == 0).sum())

Shape: (5414, 31)
Dzaleka records: 292
National records: 5122


## Imputing missing wealth index

The 292 null values in str_wealth_index all belong to Dzaleka refugee camp records. 
These records have no wealth index in the national classification because the camp 
was a separate sampling domain. We impute using the district-level mode. Since all 
292 missing records belong to dowa (camps) and that district has no non-missing wealth 
values to compute a mode from, we use the adjacent Dowa district mode as a proxy. 
The camp is located within Dowa district and the two populations share the same 
agro-ecological and economic context. The imputed value of poorest is consistent 
with the known socioeconomic profile of the Dzaleka camp population.

In [8]:
# compute district mode from records with valid wealth index
district_modes = (df[df["str_wealth_index"].notna()]
                  .groupby("str_district")["str_wealth_index"]
                  .agg(lambda x: x.mode().iloc[0]))

# dowa (camps) has no valid wealth records so use dowa district as proxy
district_modes["dowa (camps)"] = district_modes["dowa"]
print("Dowa camps assigned mode:", district_modes["dowa (camps)"])

# impute missing values
mask = df["str_wealth_index"].isnull()
df.loc[mask, "str_wealth_index"] = df.loc[mask, "str_district"].map(district_modes)

print("\nWealth index after imputation:")
print(df["str_wealth_index"].value_counts(dropna=False))
print("Remaining nulls:", df["str_wealth_index"].isnull().sum())

Dowa camps assigned mode: poorest

Wealth index after imputation:
str_wealth_index
poorest    1442
richest    1093
richer     1041
poorer      945
middle      893
Name: count, dtype: int64
Remaining nulls: 0


## Engineering child-level features

Three new features are derived from existing child-level variables.

Age bands divide the continuous age in months into five groups aligned to the 
biological windows of the first 1000 days. The 12-23 month window is where stunting 
accumulates most rapidly in Malawi and across sub-Saharan Africa. These bands allow 
models that cannot capture non-linearity, such as logistic regression, to represent 
the age-stunting relationship correctly.

Short birth interval flags any preceding interval below 24 months. The WHO identifies 
intervals under 24 months as a significant risk factor for child stunting and mortality 
through maternal nutritional depletion.

High birth order flags birth order above 4, which research in Malawi has associated 
with increased stunting risk through household resource dilution.

In [9]:
# age bands aligned to first 1000 days biological windows
age_bins   = [-1, 5, 11, 23, 35, 59]
age_labels = ["0_5m", "6_11m", "12_23m", "24_35m", "36_59m"]

df["imm_age_band"] = pd.cut(
    df["imm_child_age_months"],
    bins=age_bins,
    labels=age_labels
).astype(str)

# short birth interval flag: WHO threshold is 24 months
df["imm_short_interval"] = (df["imm_birth_interval"] < 24).astype(int)

# high birth order flag
df["imm_high_birth_order"] = (df["imm_birth_order"] > 4).astype(int)

print("Age band distribution:")
print(df["imm_age_band"].value_counts().sort_index())

print("\nShort interval flag:")
print(df["imm_short_interval"].value_counts())

print("\nHigh birth order flag:")
print(df["imm_high_birth_order"].value_counts())

Age band distribution:
imm_age_band
0_5m       586
12_23m    1176
24_35m    1032
36_59m    2019
6_11m      601
Name: count, dtype: int64

Short interval flag:
imm_short_interval
0    5123
1     291
Name: count, dtype: int64

High birth order flag:
imm_high_birth_order
0    4442
1     972
Name: count, dtype: int64


## Engineering maternal features

Two new features are derived from maternal measurements.

Maternal BMI is calculated from weight and height. Low maternal BMI below 18.5 is 
a direct measure of maternal undernutrition and a strong predictor of child stunting 
through placental insufficiency and low birth weight. A binary low BMI flag is added 
alongside the continuous value.

Maternal stunting is defined as height below 145 cm, which is the standard threshold 
used in DHS analysis for Malawi. Maternal stunting captures the intergenerational 
transmission of undernutrition and is one of the strongest known predictors of child 
stunting in sub-Saharan Africa.

Outlier capping is applied to birth interval at the 99th percentile. The maximum of 
249 months is biologically implausible for a preceding birth interval and represents 
a data entry error. Maternal weight and height extremes are reviewed but not capped 
since the ranges, while wide, are not impossible for a population survey.

In [10]:
# maternal BMI
df["und_maternal_bmi"] = (
    df["und_maternal_weight_kg"] /
    ((df["und_maternal_height_cm"] / 100) ** 2)
).round(2)

df["und_low_bmi"] = (df["und_maternal_bmi"] < 18.5).astype(int)

# maternal stunting flag
df["und_maternal_stunted"] = (df["und_maternal_height_cm"] < 145).astype(int)

# cap birth interval at 99th percentile
p99 = df["imm_birth_interval"].quantile(0.99)
df["imm_birth_interval"] = df["imm_birth_interval"].clip(upper=p99)

print("Maternal BMI summary:")
print(df["und_maternal_bmi"].describe().round(2))

print("\nLow BMI flag:")
print(df["und_low_bmi"].value_counts())

print("\nMaternal stunting flag:")
print(df["und_maternal_stunted"].value_counts())

print(f"\nBirth interval capped at: {p99:.1f} months")
print(df["imm_birth_interval"].describe().round(2))

Maternal BMI summary:
count    5414.00
mean       23.30
std         4.41
min        15.59
25%        20.55
50%        22.40
75%        24.91
max        68.07
Name: und_maternal_bmi, dtype: float64

Low BMI flag:
und_low_bmi
0    5143
1     271
Name: count, dtype: int64

Maternal stunting flag:
und_maternal_stunted
0    5315
1      99
Name: count, dtype: int64

Birth interval capped at: 142.9 months
count    5414.00
mean       55.16
std        22.89
min         8.00
25%        44.00
50%        52.00
75%        61.00
max       142.87
Name: imm_birth_interval, dtype: float64


## Collapsing religion categories and encoding categoricals

Religion has nine categories but three are very small: no religion at 41 records, 
other at 3 records, and anglican at 185. We collapse these into five groups: 
catholic, protestant (other christian plus anglican plus ccap plus seven days/baptist), 
pentecostal, muslim, and other. This reduces sparsity without losing the meaningful 
distinctions between catholics, pentecostals, and muslims which have different dietary 
and health-seeking norms in Malawi.

All categorical variables are then encoded to numeric values for modeling. Ordered 
categories use ordinal encoding that preserves rank information. Nominal categories 
use label encoding. The original string columns are retained alongside the encoded 
versions so the dataset remains interpretable.

In [11]:
# collapse religion to five groups
religion_map = {
    "catholic"          : "catholic",
    "muslim"            : "muslim",
    "pentecostal"       : "pentecostal",
    "other christian"   : "protestant",
    "ccap"              : "protestant",
    "seven days/baptist": "protestant",
    "anglican"          : "protestant",
    "no religion"       : "other",
    "other"             : "other"
}
df["str_religion_grouped"] = df["str_religion"].map(religion_map).fillna("other")

print("Religion groups:")
print(df["str_religion_grouped"].value_counts())

# ordinal encoding for ordered categories
wealth_order = {"poorest": 0, "poorer": 1, "middle": 2, "richer": 3, "richest": 4}
edu_order    = {"no education": 0, "primary": 1, "secondary": 2, "higher": 3}
age_order    = {"0_5m": 0, "6_11m": 1, "12_23m": 2, "24_35m": 3, "36_59m": 4}

df["enc_wealth_index"]   = df["str_wealth_index"].map(wealth_order)
df["enc_edu_level"]      = df["und_maternal_edu_level"].map(edu_order)
df["enc_age_band"]       = df["imm_age_band"].map(age_order)

# label encoding for nominal categories
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col, new_col in [
    ("imm_child_sex",        "enc_child_sex"),
    ("imm_size_at_birth",    "enc_size_at_birth"),
    ("imm_had_diarrhea",     "enc_had_diarrhea"),
    ("str_residence",        "enc_residence"),
    ("str_region",           "enc_region"),
    ("str_district",         "enc_district"),
    ("str_religion_grouped", "enc_religion")
]:
    df[new_col] = le.fit_transform(df[col].astype(str))
    print(f"{new_col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Religion groups:
str_religion_grouped
protestant     3204
pentecostal     771
catholic        702
muslim          693
other            44
Name: count, dtype: int64
enc_child_sex: {'female': np.int64(0), 'male': np.int64(1)}
enc_size_at_birth: {'average': np.int64(0), 'larger than average': np.int64(1), 'not_reported': np.int64(2), 'smaller than average': np.int64(3), 'very large': np.int64(4), 'very small': np.int64(5)}
enc_had_diarrhea: {'no': np.int64(0), 'yes, last two weeks': np.int64(1)}
enc_residence: {'rural': np.int64(0), 'urban': np.int64(1)}
enc_region: {'central': np.int64(0), 'northern': np.int64(1), 'southern': np.int64(2)}
enc_district: {'balaka': np.int64(0), 'blantyre': np.int64(1), 'blantyre city': np.int64(2), 'chikwawa': np.int64(3), 'chiradzulu': np.int64(4), 'chitipa': np.int64(5), 'dedza': np.int64(6), 'dowa': np.int64(7), 'dowa (camps)': np.int64(8), 'karonga': np.int64(9), 'kasungu': np.int64(10), 'likoma': np.int64(11), 'lilongwe': np.int64(12), 'lilongwe city'

## Creating the national analysis dataset and saving

The full dataset with all engineered features is saved as model_dataset_v4.parquet. 
A second file, model_dataset_national.parquet, contains only the 5122 national sample 
records with Dzaleka excluded. This is the file that all modeling notebooks will load 
by default. Dzaleka records remain available in v4 for separate analysis.

In [12]:
print("=== Final feature set ===")
print(f"Total columns: {df.shape[1]}")
print(f"Total records: {df.shape[0]}")

print("\n=== Missingness check ===")
miss = df.isnull().mean().round(3)
print(miss[miss > 0] if miss[miss > 0].any() else "No missing values")

print("\n=== Encoded feature summary ===")
enc_cols = [c for c in df.columns if c.startswith("enc_")]
print(df[enc_cols].describe().round(2))

# save full dataset including Dzaleka
for col in df.select_dtypes(["category"]).columns:
    df[col] = df[col].astype(str)

df.to_parquet(DATA_PROCESSED / "model_dataset_v4.parquet", index=False)
print("\nSaved: model_dataset_v4.parquet")
print("Shape:", df.shape)

# save national sample only
df_national = df[df["flag_dzaleka"] == 0].copy()
df_national.to_parquet(DATA_PROCESSED / "model_dataset_national.parquet", index=False)
print("\nSaved: model_dataset_national.parquet")
print("Shape:", df_national.shape)

print("\n=== All columns ===")
for col in df.columns:
    print(" ", col)

=== Final feature set ===
Total columns: 48
Total records: 5414

=== Missingness check ===
whz_score         0.008
waz_score         0.001
hemoglobin_gdl    0.250
outcome_anemia    0.250
dtype: float64

=== Encoded feature summary ===
       enc_wealth_index  enc_edu_level  enc_age_band  enc_child_sex  \
count           5414.00        5414.00       5414.00        5414.00   
mean               1.89           1.23          2.61           0.49   
std                1.49           0.60          1.36           0.50   
min                0.00           0.00          0.00           0.00   
25%                0.00           1.00          2.00           0.00   
50%                2.00           1.00          3.00           0.00   
75%                3.00           2.00          4.00           1.00   
max                4.00           3.00          4.00           1.00   

       enc_size_at_birth  enc_had_diarrhea  enc_residence  enc_region  \
count            5414.00           5414.00        54